# RSNA Knee Abnormality Detection — Starter EDA

A guided walkthrough of the competition: what the data looks like, how the pieces (images + text) fit together, what you're predicting, and how it's scored.

**Where to run this:** the training data is 569.76 GB of DICOM files — see the project `README.md` for why we never download that locally. Run this notebook **on Kaggle** with the competition dataset attached ("Add Data" -> `rsna-knee-abnormality-detection`), where everything below runs against the real data. Locally, only the small CSV metadata files plus one sample series (downloaded for this demo) are used.

## The task
For each knee MRI **study**, predict a probability for each of 12 findings:

`ACL, MCL, Medial Meniscus, Lateral Meniscus, Medial OA, Lateral OA, PF OA, Effusion, Synovitis, Baker's, Contusion, Fracture`

Scored by **macro-averaged AUC-ROC** across those 12 columns (AUC per column, then averaged — implemented by hand further down so you can validate locally before submitting).

## The data
- `train.csv` — one row per study: `StudyInstanceUID`, `PatientSex`, `Report` (free-text radiology report, may be non-English), and the 12 binary labels. **Only a small subset of studies have all 12 labels filled in** — the rest are report-only.
- `train_series.csv` — one row per MRI series (a study has several): plane (`Anatomical_Plane`: Sagittal/Coronal/Axial) and sequence-type flags (`Fluid_Sensitive`, `Fat_Suppression`).
- `train_series/<StudyInstanceUID>/<SeriesInstanceUID>/*.dcm` — the actual DICOM slices, ~20-45 per series (median 30).
- `sample_submission.csv` — the exact output format expected.

## Why this matters for strategy
Two things fall out of the data itself:
1. Most studies are **report-only** (no structured labels) -> mining weak labels from `Report` text is the cheapest way to grow the training set.
2. Each study has **multiple series** at full resolution -> for a laptop + free-Kaggle-GPU budget, we deliberately pick 1-2 representative series and subsample slices rather than using everything (see `README.md`).

Let's look at the real numbers.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 120)

CANDIDATES = [
    Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),  # Kaggle Notebook with dataset attached
    Path("../data"),                                         # local fallback (small files only, gitignored)
]

DATA_DIR = next((c for c in CANDIDATES if (c / "train.csv").exists()), None)

if DATA_DIR is None:
    print(
        "No dataset found.\n"
        "- On Kaggle: click 'Add Data' and attach 'rsna-knee-abnormality-detection'.\n"
        "- Locally: download just the small metadata files, e.g.\n"
        "    kaggle competitions download -c rsna-knee-abnormality-detection -f train.csv -p ../data\n"
        "    kaggle competitions download -c rsna-knee-abnormality-detection -f train_series.csv -p ../data\n"
        "  Do NOT download train_series/ in full locally -- it's 569.76 GB."
    )
else:
    print(f"Using data from: {DATA_DIR}")

## 1. Study-level labels (`train.csv`)

In [ ]:
LABEL_COLS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture",
]

train = pd.read_csv(DATA_DIR / "train.csv") if DATA_DIR else None
print(train.shape if train is not None else "no data loaded")
train.head() if train is not None else None

### How many studies actually have structured labels?

In [ ]:
if train is not None:
    has_labels = train[LABEL_COLS].notna().all(axis=1)
    print(f"Studies with all 12 labels filled in: {has_labels.sum()} / {len(train)} "
          f"({has_labels.mean():.1%})")
    print(f"Report-only studies (no structured labels): {(~has_labels).sum()}")

### Label prevalence (among labeled studies)

In [ ]:
if train is not None and has_labels.sum() > 0:
    labeled = train.loc[has_labels, LABEL_COLS].astype(float)
    prevalence = labeled.mean().sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(8, 5))
    prevalence.plot.barh(ax=ax)
    ax.set_xlabel("Fraction of labeled studies positive")
    ax.set_title("Label prevalence")
    plt.tight_layout()
    plt.show()

    prevalence
else:
    print("No labeled studies in this slice of data -- expected if train.csv here is mostly report-only.")

### A peek at the report text

In [ ]:
if train is not None:
    for _, row in train.dropna(subset=["Report"]).head(3).iterrows():
        print("=" * 80)
        print("StudyInstanceUID:", row["StudyInstanceUID"])
        print("Positive labels:", [c for c in LABEL_COLS if row.get(c) == 1])
        print("-" * 80)
        print(str(row["Report"])[:800])

This is the raw material for weak-label mining: reports mention findings in free text (in several languages) even when the structured columns are empty. A keyword/rule pass (or a small LLM prompt) over `Report` for report-only studies is the cheapest way to expand supervision -- that's the next notebook.

## 2. Series-level metadata (`train_series.csv`)

In [ ]:
series = pd.read_csv(DATA_DIR / "train_series.csv") if DATA_DIR else None
series.head() if series is not None else None

In [ ]:
if series is not None:
    per_study = series.groupby("StudyInstanceUID").size()
    print(per_study.describe())

    fig, ax = plt.subplots(figsize=(6, 4))
    per_study.plot.hist(bins=30, ax=ax)
    ax.set_xlabel("Series per study")
    ax.set_title("How many series does a study have?")
    plt.tight_layout()
    plt.show()

In [ ]:
if series is not None:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    series["Anatomical_Plane"].value_counts().plot.bar(ax=axes[0], title="Anatomical_Plane")
    series["Fluid_Sensitive"].value_counts().plot.bar(ax=axes[1], title="Fluid_Sensitive")
    series["Fat_Suppression"].value_counts().plot.bar(ax=axes[2], title="Fat_Suppression")
    plt.tight_layout()
    plt.show()

This is where the 'shrink the input' strategy comes from: a study might have 6+ series across planes and sequence types, but a sagittal fluid-sensitive series (`Anatomical_Plane == 'Sagittal'`, `Fluid_Sensitive == 1`) is usually the single most informative one for ligament/meniscus findings. Picking 1-2 series per study instead of all of them is what makes training fit a free GPU quota.

## 3. Looking at an actual DICOM series

On Kaggle, every study's DICOMs live under `train_series/<StudyInstanceUID>/<SeriesInstanceUID>/*.dcm`. Locally we don't have that (569.76 GB) -- this repo has one **sample series** downloaded from the public `test_series/` example data purely to make this section runnable. The loading code is identical either way; only the folder it points at changes.

In [ ]:
import pydicom


def load_series_volume(series_dir: Path) -> np.ndarray:
    """Read every .dcm slice in a series folder, sorted by InstanceNumber, into one array."""
    files = sorted(
        series_dir.glob("*.dcm"),
        key=lambda p: pydicom.dcmread(p, stop_before_pixels=True).InstanceNumber,
    )
    return np.stack([pydicom.dcmread(f).pixel_array for f in files])


def find_example_series_dir(data_dir: Path) -> Path | None:
    for subset in ("train_series", "test_series"):
        root = data_dir / subset
        if not root.exists():
            continue
        for study_dir in root.iterdir():
            for series_dir in study_dir.iterdir():
                if any(series_dir.glob("*.dcm")):
                    return series_dir
    return None

In [ ]:
example_series_dir = find_example_series_dir(DATA_DIR) if DATA_DIR else None

if example_series_dir is not None:
    volume = load_series_volume(example_series_dir)
    print("Series:", example_series_dir)
    print("Volume shape (slices, H, W):", volume.shape)

    n_show = min(8, volume.shape[0])
    idxs = np.linspace(0, volume.shape[0] - 1, n_show).astype(int)
    fig, axes = plt.subplots(1, n_show, figsize=(2 * n_show, 3))
    for ax, i in zip(axes, idxs):
        ax.imshow(volume[i], cmap="gray")
        ax.set_title(f"slice {i}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No DICOM series found on disk -- run this on Kaggle with the dataset attached to see real slices.")

## 4. The evaluation metric: macro AUC-ROC

In [ ]:
from sklearn.metrics import roc_auc_score


def macro_auc(y_true: pd.DataFrame, y_pred: pd.DataFrame, label_cols=LABEL_COLS) -> float:
    """Average AUC-ROC across the 12 label columns -- exactly how the leaderboard scores you."""
    scores = [roc_auc_score(y_true[c], y_pred[c]) for c in label_cols]
    return float(np.mean(scores))

In [ ]:
if train is not None and has_labels.sum() > 0:
    labeled_df = train.loc[has_labels, LABEL_COLS].reset_index(drop=True).astype(float)
    rng = np.random.default_rng(0)
    dummy_preds = pd.DataFrame(
        rng.uniform(size=(len(labeled_df), len(LABEL_COLS))),
        columns=LABEL_COLS,
    )
    print(
        "Macro AUC of random predictions (sanity check, expect ~0.5):",
        round(macro_auc(labeled_df, dummy_preds), 3),
    )
else:
    print("No labeled rows available in this slice of data to sanity-check the metric against.")

A submission is just `sample_submission.csv`'s shape filled in with your own probabilities per column -- `macro_auc()` above is the exact function the leaderboard is (functionally) running on your predictions, so use it for local validation on a held-out split before ever submitting.

## What you should take away from this notebook
- 12-label multi-label problem, macro AUC-ROC scored, code-competition with a 9h inference cap.
- Most studies are report-only -> text-based weak labeling matters a lot (see the label-mining notebook, next).
- Each study has multiple series; picking a small, informative subset per study is what makes this tractable on a free GPU budget.
- `macro_auc()` above is your local validation metric -- use it on a held-out split before ever submitting.

**Next up:** a label-mining notebook that turns `Report` text into weak labels for the report-only studies, and a baseline training notebook using the reduced-series-per-study approach from the README.